# JARL Showcase Pipeline launcher

Notebook para lanzar el pipeline showcase (`jarl.showcase`) desde Jupyter: validar el plan, ejecutar smoke/medium/large y abrir el resultado en Runner Lab.

## Requisitos

```bash
uv sync --extra app
```

## Uso

1. Ajusta **configuración** en la celda siguiente (`SCALE`, `NAME`, `EXPERIMENTS_ROOT`, `ALGORITHM`, flags).
2. Ejecuta **dry-run** para validar el plan sin entrenar.
3. Ejecuta **run** para entrenar (smoke ~5–15 min; medium horas).
4. Opcional: arranca Runner Lab con `src/jarl/app/runner_lab_launcher.ipynb`.

In [ ]:
import os
import subprocess
from pathlib import Path

from jarl.experiments.paths import resolve_workspace_root
from jarl.utils.env import load_project_env

# --- Configuración (edita aquí) ---
SCALE = "smoke"  # smoke | medium | large
NAME = "jarl-smoke1.3-ppo"
ALGORITHM = "ppo"  # ppo | ppo_gru
# None → JARL_EXPERIMENTS_ROOT en .env o results/dags
EXPERIMENTS_ROOT = None
DRY_RUN = False
REUSE = False
FROM_NODE = None  # p. ej. "empty5_lr_low" para reanudar
FORCE = False
WANDB_ONLINE = True
WANDB_PROJECT = "jarl-smoke1.3-ppo"  # proyecto en wandb.ai
VERBOSE = True

APP_ENTRY = Path("src/jarl/app/app.py")


def find_repo_root(start: Path) -> Path:
    """Find JARL repository root from a notebook working directory."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").is_file() and (path / APP_ENTRY).is_file():
            return path
    raise FileNotFoundError(f"No se encontró la raíz del repo JARL (se esperaba {APP_ENTRY} junto a pyproject.toml).")


repo = find_repo_root(Path.cwd())
load_project_env(start=repo)
workspace = resolve_workspace_root(override=EXPERIMENTS_ROOT)
exp_dir = workspace / NAME

print(f"Repo: {repo}")
print(f"Escala: {SCALE}")
print(f"Algoritmo: {ALGORITHM}")
print(f"Workspace: {workspace}")
print(f"Experimento: {exp_dir}")

print(f"WANDB_API_KEY cargada: {bool(os.environ.get('WANDB_API_KEY'))}")
print(f"WANDB project: {WANDB_PROJECT}")

In [ ]:
def build_showcase_command(*, dry_run: bool = False) -> list[str]:
    """Build the ``python -m jarl.showcase`` argv list."""
    cmd = [
        "uv",
        "run",
        "python",
        "-m",
        "jarl.showcase",
        "--name",
        NAME,
        "--workspace",
        str(workspace),
        "--scale",
        SCALE,
        "--algorithm",
        ALGORITHM,
    ]
    if dry_run:
        cmd.append("--dry-run")
    if REUSE:
        cmd.append("--reuse")
    if FROM_NODE:
        cmd.extend(["--from-node", FROM_NODE])
    if FORCE:
        cmd.append("--force")
    if WANDB_ONLINE:
        cmd.append("--wandb-online")
    if WANDB_PROJECT:
        cmd.extend(["--wandb-project", WANDB_PROJECT])
    if VERBOSE:
        cmd.append("-v")
    return cmd


def run_showcase(*, dry_run: bool = False) -> int:
    """Run showcase subprocess with streamed stdout/stderr."""
    cmd = build_showcase_command(dry_run=dry_run)
    print("Commando:", " ".join(cmd))
    completed = subprocess.run(cmd, cwd=repo, check=False)
    return int(completed.returncode)

## Dry-run

Valida YAML, orden topológico, transfer y divisibilidad de minibatch **sin JAX ni entrenamiento**.

In [ ]:
exit_code = run_showcase(dry_run=True)
print(f"exit_code={exit_code}")

## Ejecución

Entrena el subárbol/árbol según la escala. Logs en `{exp_dir}/showcase.log`. Al terminar se escribe `showcase_summary.json`.

In [ ]:
if DRY_RUN:
    print("DRY_RUN=True: cambia a False en la celda de configuración para entrenar.")
else:
    exit_code = run_showcase(dry_run=False)
    print(f"exit_code={exit_code}")
    if exit_code == 0:
        print(f"Experimento listo: {exp_dir}")
        print("Runner Lab: uv run streamlit run src/jarl/app/app.py")

## Estado y resumen

Lee `showcase_run.json` (incremental) y `showcase_summary.json` (panel en Inicio de Runner Lab).

In [ ]:
import json

for filename in ("showcase_run.json", "showcase_summary.json", "showcase.log"):
    path = exp_dir / filename
    print(f"--- {filename} ({'ok' if path.is_file() else 'missing'}) ---")
    if not path.is_file():
        continue
    if filename.endswith(".json"):
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(json.dumps(payload, indent=2)[:4000])
    else:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("\n".join(lines[-40:]))